# Generating Data

In [3]:
import cv2

In [5]:
def generate_dataset():
    face_classifier = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

    def face_cropped(img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_classifier.detectMultiScale(gray, 1.3, 5)

        if len(faces) == 0:
            return None
        for (x, y, w, h) in faces:
            cropped_face = img[y:y+h, x:x+w]
            return cropped_face

    cap = cv2.VideoCapture(0)
    id = 1
    img_id = 0

    while True:
        ret, frame = cap.read()
        face = face_cropped(frame)
        if face is not None:
            img_id += 1
            face = cv2.resize(face, (200, 200))
            face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
            file_name_path = "data/user." + str(id) + "." + str(img_id) + ".jpg"
            cv2.imwrite(file_name_path, face)
            cv2.putText(face, str(img_id), (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (0, 255, 0), 2)

            cv2.imshow("Cropped face", face)

        if cv2.waitKey(1) == 13 or int(img_id) == 200:
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Collecting samples is completed....")

generate_dataset()


# Train the Classifier and Save it

In [5]:
#pip install pillow

In [7]:
import numpy as np
from PIL import Image
import os

In [9]:
def train_classifier(data_dir):
    path = [os.path.join(data_dir, f) for f in os.listdir(data_dir)]
    
    faces = []
    ids = []
    
    for image in path:
        img = Image.open(image).convert('L')
        imageNp = np.array(img, 'uint8')
        id = int(os.path.split(image)[1].split(".")[1])
        
        faces.append(imageNp)
        ids.append(id)
        
    ids = np.array(ids)
    
    # Train and save classifier
    clf = cv2.face.LBPHFaceRecognizer_create()
    clf.train(faces, ids)
    clf.write("classifier.xml")

train_classifier("data")

# Face Feature Detection

In [ ]:
import cv2

def draw_boundary(img, classifier, scaleFactor, minNeighbors, color, text, clf):
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    features = classifier.detectMultiScale(gray_img, scaleFactor, minNeighbors)

    for (x, y, w, h) in features:
        cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
        id, pred = clf.predict(gray_img[y:y+h, x:x+w])
        confidence = int(100 * (1 - pred / 300))

        if confidence > 70:
            if id == 1:
                cv2.putText(img, "Surya", (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 1)
        else:
            cv2.putText(img, "UNKNOWN", (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 1)

    return img

# Load classifiers
faceCascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier(r"C:\Users\surya\Downloads\Telegram Desktop\haar-cascade-files-master (2)\haarcascade_eye.xml")
smile_cascade = cv2.CascadeClassifier(r"C:\Users\surya\Downloads\Telegram Desktop\haar-cascade-files-master (2)\haarcascade_smile.xml")
body_cascade = cv2.CascadeClassifier(r"C:\Users\surya\Downloads\Telegram Desktop\haar-cascade-files-master (2)\haarcascade_fullbody.xml")

# Load trained face recognition model
clf = cv2.face.LBPHFaceRecognizer_create()
clf.read("classifier.xml")

# Start webcam
video_capture = cv2.VideoCapture(0)
if not video_capture.isOpened():
    print("Error: Camera not accessible")
    exit()

while True:
    ret, img = video_capture.read()
    if not ret:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Draw face and recognition
    img = draw_boundary(img, faceCascade, 1.3, 6, (0, 0, 255), "Face", clf)

    # Detect body
    bodies = body_cascade.detectMultiScale(gray, 1.1, 5)
    for (a, b, c, d) in bodies:
        cv2.rectangle(img, (a, b), (a + c, b + d), (0, 0, 255), 1)

    # Detect faces again for smile/eye detection
    faces = faceCascade.detectMultiScale(gray, 1.3, 5)
    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = img[y:y+h, x:x+w]

        # Detect smiles
        smiles = smile_cascade.detectMultiScale(roi_gray, 1.7, 22)
        for (sx, sy, sw, sh) in smiles:
            cv2.rectangle(roi_color, (sx, sy), (sx + sw, sy + sh), (0, 0, 255), 1)

        # Detect eyes
        eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 10)
        for (ex, ey, ew, eh) in eyes:
            cv2.rectangle(roi_color, (ex, ey), (ex + ew, ey + eh), (0, 0, 255), 1)

    cv2.imshow("Face + Body + Eye + Smile Detection", img)

    if cv2.waitKey(1) == 27:  # ESC key
        break

video_capture.release()
cv2.destroyAllWindows()
